In [5]:
"""
train_cnn_eeg_features.py

Usage:
    python train_cnn_eeg_features.py

Notes:
- Expects a CSV file named "EEG_features.csv" in the same folder, or change DATA_PATH.
- The CSV must include a label column named 'label' (or 'target' or 'y'). If your
  label column has a different name, either rename it to 'label' or set LABEL_COL below.
"""

import os
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, utils

# -------------------------
# Config
# -------------------------
DATA_PATH = "EEG_features.csv"   # change if your CSV has a different name/path
LABEL_COL_CANDIDATES = ["label", "target", "y"]  # common label names
RANDOM_SEED = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.15   # fraction of training data to use for validation (or you can use a separate split)
BATCH_SIZE = 32
EPOCHS = 200
MODEL_SAVE = "best_eeg_cnn.h5"

# -------------------------
# Reproducibility
# -------------------------
def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(RANDOM_SEED)

# -------------------------
# Load data
# -------------------------
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"CSV file not found at '{DATA_PATH}'. Put the CSV there or change DATA_PATH.")

df = pd.read_csv(DATA_PATH)
print("Loaded CSV shape:", df.shape)
print("Columns:", df.columns.tolist())

# Detect label column
label_col = None
for cand in LABEL_COL_CANDIDATES:
    if cand in df.columns:
        label_col = cand
        break

if label_col is None:
    raise ValueError(
        "No label column found. Please add a column named 'label' (or 'target'/'y') containing class labels."
    )

# Separate features and labels
X = df.drop(columns=[label_col])
y = df[label_col].values

# Keep only numeric features
X = X.select_dtypes(include=[np.number])
if X.shape[1] == 0:
    raise ValueError("No numeric features found. Make sure your CSV contains numeric feature columns.")

print("Using numeric feature columns:", X.columns.tolist())
print("Feature matrix shape:", X.shape)

# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)
num_classes = len(le.classes_)
print("Classes found (encoded):", dict(zip(le.classes_, range(num_classes))))

# If binary classification, Keras will use sigmoid; for multi-class, softmax.
is_binary = (num_classes == 2)

# -------------------------
# Train/test split
# -------------------------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X.values, y_enc, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_enc
)
# Further split training into train and validation
val_fraction_of_train = VAL_SIZE
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=val_fraction_of_train, random_state=RANDOM_SEED, stratify=y_train_full
)

print("Train/Val/Test shapes:", X_train.shape, X_val.shape, X_test.shape)

# -------------------------
# Scaling
# -------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# -------------------------
# Reshape for Conv1D
# We treat the feature vector as a 1D "signal" with channels=1
# Input shape: (timesteps = n_features, channels=1)
# -------------------------
n_features = X_train.shape[1]
X_train = X_train.reshape((-1, n_features, 1))
X_val = X_val.reshape((-1, n_features, 1))
X_test = X_test.reshape((-1, n_features, 1))
input_shape = (n_features, 1)
print("Reshaped input shape for Conv1D:", input_shape)

# -------------------------
# Build CNN model (Conv1D)
# -------------------------
def build_conv1d_model(input_shape, num_classes, base_filters=64, dropout_rate=0.35):
    inputs = layers.Input(shape=input_shape)
    x = layers.GaussianNoise(0.1)(inputs)  # small augmentation
    x = layers.Conv1D(filters=base_filters, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(filters=base_filters, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Conv1D(filters=base_filters*2, kernel_size=5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)

    if num_classes == 2:
        outputs = layers.Dense(1, activation="sigmoid")(x)
    else:
        outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    return model

model = build_conv1d_model(input_shape, num_classes)
model.summary()

# -------------------------
# Compile
# -------------------------
if is_binary:
    loss = "binary_crossentropy"
    metrics = ["accuracy"]
else:
    loss = "sparse_categorical_crossentropy"
    metrics = ["accuracy"]

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss=loss,
              metrics=metrics)

# -------------------------
# Class weights (helpful for imbalanced data)
# -------------------------
from sklearn.utils.class_weight import compute_class_weight
class_weights = None
try:
    classes = np.unique(y_train)
    cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weights = {int(c): float(w) for c, w in zip(classes, cw)}
    print("Computed class weights:", class_weights)
except Exception as e:
    print("Could not compute class weights:", e)
    class_weights = None

# -------------------------
# Callbacks
# -------------------------
es = callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=1)
rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6, verbose=1)
mc = callbacks.ModelCheckpoint(MODEL_SAVE, monitor="val_loss", save_best_only=True, verbose=1)

# -------------------------
# Train
# -------------------------
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[es, rlr, mc],
    class_weight=class_weights,
    verbose=2
)

# -------------------------
# Evaluate
# -------------------------
print("\n--- Evaluating on test set ---")
test_preds = model.predict(X_test)

if is_binary:
    y_pred = (test_preds.ravel() >= 0.5).astype(int)
else:
    y_pred = np.argmax(test_preds, axis=1)

acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

# -------------------------
# Save scaler and label encoder for future inference
# -------------------------
import joblib
joblib.dump(scaler, "scaler.joblib")
joblib.dump(le, "label_encoder.joblib")

print("\nSaved model to:", MODEL_SAVE)
print("Saved scaler to: scaler.joblib")
print("Saved label encoder to: label_encoder.joblib")

# -------------------------
# Tips if accuracy is low (<0.80)
# -------------------------
print("\nTIPS: If accuracy is below desired threshold (0.80):")
print("- Ensure labels are correct and balanced. Consider more data or augmentations.")
print("- Use per-channel features (e.g., Fp1_alpha) rather than single averages if available.")
print("- Try hyperparameter tuning (filters, kernel size, learning rate).")
print("- Try classical ML models (XGBoost, RandomForest) as baseline.")
print("- Use cross-validation and ensembling.")


Loaded CSV shape: (84, 6)
Columns: ['filename', 'delta_power', 'theta_power', 'alpha_power', 'beta_power', 'gamma_power']


ValueError: No label column found. Please add a column named 'label' (or 'target'/'y') containing class labels.